# Entrenamiento de 3DMambaIPF en Google Colab (T4 / A100)

Este cuaderno ejecuta en Colab lo mismo que `train_mamba.py` de la rama `3dmamba-branch`:

1. `prepare`: corta las escenas LiDAR en ventanas de 100 m y las reparte en train/val. Deja fuera el patch de evaluación de `main.py`.
2. `train --task classifier`: fase 1, poda de outliers → `mamba_classifier.pt`
3. `train --task regressor`: fase 2, desplazamientos → `mamba_regressor.pt`

El código del repositorio se usa **sin modificarlo**: el cuaderno lo clona e importa sus funciones. Solo cambia algunos valores de `config.py` en memoria y la configuración de ejecución.

## Antes de empezar

**1. Entorno:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución*.
- Para el preset `a100` (por defecto): **GPU A100** y **RAM alta** (Colab Pro).
- Para el preset `t4` (corrida preliminar): una T4 basta.

**2. Datos en Google Drive** (la carpeta `data/` está en `.gitignore`, así que no viene con el `git clone`). Sube **solo estos 3 archivos** (~420 MB), con los mismos nombres de carpeta que en tu `data/` local:

```
MyDrive/3d-denoising/
└── data/
    ├── Sunset_Crater_Volcano_National_Monument_1M/points.laz    # ~4 MB: patch de evaluación (solo para excluirlo del entrenamiento)
    ├── Northern_California_Coast_40M/points.laz                 # ~170 MB: entrenamiento
    └── Sunset_Crater_Volcano_National_Monument_80M/points.laz   # ~250 MB: entrenamiento
```

No subas los archivos de 400k/1M de las otras escenas (dan muy pocas ventanas de 100 m) ni `San_Isabel_NF_CO_Debris_100M`: queda fuera de esta corrida preliminar (necesita ~10 GB de RAM para prepararse y su densidad, 5,5 pt/m², es la más lejana a la del patch de evaluación, 18 pt/m²).

El cuaderno crea `mamba_windows/` (ventanas ya preparadas) y `checkpoints/` en esa misma carpeta de Drive. Así sobreviven a una desconexión, y en las siguientes sesiones se salta `prepare`.

**3. Presets** (paso 5), comparados con los valores por defecto de `config.py`:

| Parámetro | `config.py` | `t4` (preliminar) | `a100` |
|---|---|---|---|
| Escenas | 3 | 2 (40M + 80M) | 2 (40M + 80M) |
| `epochs` (máximo, ajustado al presupuesto) | 100 | 40 | 60 |
| `items_per_epoch` / `val_items` | 500 / 60 | 200 / 40 | 300 / 60 |
| `patch_size` (puntos por parche) | 1000 | 512 | 1000 |
| `samples_per_item` (parches por lote) | 8 | 4 | 8 |
| `num_modules` del regresor | 4 | 2 | 4 |
| Presupuesto clasificador / regresor | — | 60 / 120 min | 75 / 180 min |
| Arranque | — | aleatorio | desde los checkpoints `t4` (`WARM_START`) |
| Salida | — | `mamba_*.pt` | `mamba_*_a100.pt` (no pisa los `t4`) |

La prueba corta (paso 7) mide la duración real de una época y reduce las épocas si el máximo no cabe en el presupuesto.

`patch_size` y `num_modules` se guardan dentro del checkpoint, y `main.py` los lee de ahí al hacer inferencia. No hay que tocar esos valores en `config.py` local.

**4. Orden recomendado en la sesión A100** (la sesión T4 ya dejó en Drive las ventanas y `mamba_classifier.pt` / `mamba_regressor.pt`):
1. Pasos 1-6. `prepare` se salta solo porque las ventanas ya están en Drive.
2. **Paso 6b**: diagnóstico del umbral del clasificador T4, que tarda minutos. Según el resultado, decide si sacas `"classifier"` de `TRAIN_TASKS` y vuelves a ejecutar el paso 5.
3. Paso 7 (prueba corta), paso 9 (regresor) y, si sigue en `TRAIN_TASKS`, paso 8 (clasificador).
4. Pasos 10-11 (curvas y checkpoints). Si reentrenaste el clasificador, repite el 6b con `CALIBRATE = OUT_CKPT["classifier"]`.

## 1. Comprobar la GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import sys, torch
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.version.cuda)
assert torch.cuda.is_available(), "No hay GPU: Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU"

## 2. Dependencias

- No se usan las versiones del README (`torch 2.1.2` + `mamba-ssm 1.1.3`): no hay wheels para el Python de Colab (3.12) y habría que compilar. Se usa el **torch que ya trae Colab** con `mamba-ssm` 2.x y `causal-conv1d`, que publican wheels precompiladas para torch 2.6-2.10 (CUDA 12, Python 3.10-3.13). `models/mamba_ipf/backbone.py` importa `mamba_ssm.modules.mamba_simple.Mamba`, que en 2.x tiene la misma firma y los mismos nombres de parámetros. Por eso los checkpoints siguen siendo compatibles con la implementación de referencia de `mamba_fallback.py`, que se usa en local sin CUDA.
- `--no-build-isolation` hace que el instalador use el torch ya instalado para elegir la wheel. Si no encuentra una, **compila desde fuente** (30-60 min). Por eso la celda se detiene antes si tu versión de torch no tiene wheels.
- `open3d` no hace falta para entrenar y no se instala.

In [ ]:
import re, subprocess, sys, torch

torch_mm = tuple(int(v) for v in re.match(r"(\d+)\.(\d+)", torch.__version__).groups())
WHEEL_TORCH = {(2, 6), (2, 7), (2, 8), (2, 9), (2, 10)}   # versiones con wheels de mamba-ssm 2.3.x / causal-conv1d 1.6+

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip_install("laspy[lazrs]")
if torch_mm not in WHEEL_TORCH:
    raise RuntimeError(
        f"torch {torch.__version__} no tiene wheels precompiladas de mamba-ssm: ejecuta la celda de RESPALDO de abajo, "
        "reinicia la sesión y vuelve a empezar desde la celda 1."
    )
pip_install("--no-build-isolation", "causal-conv1d>=1.4", "mamba-ssm>=2.2")
!pip list 2>/dev/null | grep -iE "^(torch|mamba|causal|laspy|lazrs|numpy) "

**RESPALDO (solo si la celda anterior lo pide).** Instala un torch con wheels de mamba-ssm. Después: *Entorno de ejecución → Reiniciar sesión* y vuelve a empezar desde la celda 1. Quita el `#` para ejecutarla.

In [ ]:
# !pip install -q "torch==2.9.*" --index-url https://download.pytorch.org/whl/cu126

## 3. Google Drive y código del repositorio

El código se clona de GitHub (rama `3dmamba-branch`), **no desde tu equipo**. Colab solo ve lo que esté commiteado y pusheado. Si cambias algo en local (por ejemplo `config.py` o `models/mamba_ipf/`), haz `git push` antes de ejecutar esta celda, o cambia el valor en memoria en el paso 5. El hash que imprime la celda debe coincidir con tu `git log -1` local.

In [ ]:
import os, sys
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/3d-denoising")   # <- cámbialo si usas otra carpeta
DRIVE_DATA = DRIVE_ROOT / "data"
DRIVE_WINDOWS = DRIVE_ROOT / "mamba_windows"
DRIVE_CKPT = DRIVE_ROOT / "checkpoints"
DRIVE_CKPT.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/Yellow2392/DL-Point-Cloud-Denoising-Pipeline.git"
BRANCH = "3dmamba-branch"
REPO_DIR = Path("/content/3d-denoising-pipeline")

if not REPO_DIR.exists():
    !git clone -q --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
!git -C {REPO_DIR} log -1 --oneline

os.chdir(REPO_DIR)                       # config.py usa rutas relativas (data/, checkpoints/, analytics/)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Archivos .laz en Drive:", sorted(str(p.relative_to(DRIVE_DATA)) for p in DRIVE_DATA.glob("*/points.laz")))

## 4. Verificar los kernels CUDA de Mamba

Se compara el bloque `Mamba` de `mamba_ssm` (CUDA) con la implementación de referencia del repo, usando los mismos pesos. Si coinciden, el entrenamiento usa el camino rápido y el checkpoint resultante se podrá cargar en local con la referencia.

In [ ]:
import torch
from mamba_ssm.modules.mamba_simple import Mamba as MambaCUDA
from models.mamba_ipf.mamba_fallback import Mamba as MambaRef

torch.manual_seed(0)
fast = MambaCUDA(d_model=48).cuda().eval()
ref = MambaRef(d_model=48).cuda().eval()
ref.load_state_dict(fast.state_dict())          # mismos nombres y formas de parámetros
x = torch.randn(2, 256, 48, device="cuda")
with torch.no_grad():
    diff = (fast(x) - ref(x)).abs().max().item()
print(f"Diferencia máxima CUDA vs. referencia: {diff:.2e}")
assert diff < 1e-3, "Los kernels de mamba_ssm no coinciden con la referencia: revisa la instalación"
del fast, ref, x

## 5. Configuración del entrenamiento

Todo se ajusta aquí. Los cambios a `cfg` solo existen en esta sesión, y `config.py` no se modifica.

- **`PRESET`**:
  - `"t4"` es la corrida preliminar (T4, 2 módulos, parche de 512 puntos, 4 parches por lote).
  - `"a100"` vuelve a la capacidad original de `config.py` (4 módulos, parche de 1000 puntos, 8 parches por lote), activa TF32 y usa más procesos de carga.
- **`WARM_START`**: parte de los checkpoints de la corrida T4 (`mamba_classifier.pt` / `mamba_regressor.pt`) en lugar de pesos aleatorios.
  - El regresor carga sus 2 módulos entrenados. Los 2 nuevos empiezan sin mover los puntos, porque su capa de salida arranca en cero.
  - Los resultados se guardan con el sufijo `_a100`, así que los checkpoints T4 no se sobrescriben.
- **`TRAIN_TERRAINS`**: 40M + 80M, las mismas escenas que en la corrida T4, para que la comparación entre ambas sea justa. Cada escena conserva la semilla de su posición en `config.MAMBA_TRAIN_TERRAINS`.
- **`TIME_BUDGET_MIN`**: minutos de entrenamiento por modelo. El paso 7 ajusta las épocas para no pasarse, con el máximo del preset como tope.
- **`cfg.MAMBA_TRAIN_COMBINATIONS`**: degradaciones vistas en entrenamiento (ahora `[["gaussian", "global_outliers"]]`). Conviene que incluya lo que vayas a evaluar en `main.py`.
- Si sale *CUDA out of memory*, baja `samples_per_item` o `patch_size`.

In [ ]:
import argparse, os
import config as cfg
import train_mamba as tm

PRESET = "a100"        # "t4" (corrida preliminar) o "a100"
WARM_START = True      # partir de mamba_classifier.pt / mamba_regressor.pt de la corrida T4
TRAIN_TASKS = ["classifier", "regressor"]   # quita "classifier" si el paso 6b muestra que basta con ajustar el umbral

TRAIN_TERRAINS = [
    "Northern_California_Coast_40M",
    "Sunset_Crater_Volcano_National_Monument_80M",
    # "San_Isabel_NF_CO_Debris_100M",   # fuera de la corrida preliminar (~10 GB de RAM en prepare)
]
cfg.MAMBA_WINDOWS_DIR = DRIVE_WINDOWS                   # ventanas en Drive: se preparan una sola vez
cfg.MAMBA_WINDOWS_PER_SPLIT = {"train": 40, "val": 6}  # igual que config.py (las ventanas pesan poco)
# cfg.MAMBA_TRAIN_COMBINATIONS = [["gaussian", "global_outliers"], ["gaussian", "laplacian", "clustered_outliers"]]

COMMON = dict(batch_items=1, k_neighbors=cfg.MAMBA_K_NEIGHBORS, lr=cfg.MAMBA_LR, noise_decay=cfg.MAMBA_NOISE_DECAY)
PRESETS = {
    # epochs es un máximo: el paso 7 lo reduce si no cabe en TIME_BUDGET_MIN
    "t4": dict(train=dict(COMMON, epochs=40, items_per_epoch=200, val_items=40, patch_size=512, samples_per_item=4,
                          num_workers=2),
               modules=2, budget={"classifier": 60, "regressor": 120}, suffix=""),
    "a100": dict(train=dict(COMMON, epochs=60, items_per_epoch=300, val_items=60, patch_size=1000, samples_per_item=8,
                            num_workers=min(8, os.cpu_count() or 2)),
                 modules=4, budget={"classifier": 75, "regressor": 180}, suffix="_a100"),
}
LIGHT = PRESETS[PRESET]["train"]
REGRESSOR_MODULES = PRESETS[PRESET]["modules"]
TIME_BUDGET_MIN = dict(PRESETS[PRESET]["budget"])
EPOCHS = {task: LIGHT["epochs"] for task in TIME_BUDGET_MIN}   # el paso 7 lo ajusta al presupuesto
OUT_CKPT = {task: DRIVE_CKPT / f"mamba_{task}{PRESETS[PRESET]['suffix']}.pt" for task in TIME_BUDGET_MIN}

if PRESET == "a100":
    # TF32 en las capas lineales: mucho más rápido en Ampere, con precisión de sobra para entrenar
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

def warm_start_file(task):
    # Adapta un checkpoint propio al formato que espera load_pretrained_ipf (el del 3DMambaIPF original):
    #   regresor: claves "feature_nets.i.*" -> se cargan tal cual (módulos que no existan en el origen quedan nuevos)
    #   clasificador: "net.*" -> "feature_nets.0.*" (load_pretrained_ipf lo vuelve a mapear a "net.*")
    src = DRIVE_CKPT / f"mamba_{task}.pt"
    if not WARM_START:
        return None
    if not src.exists():
        print(f"AVISO: no existe {src}; {task} empieza desde cero")
        return None
    ckpt = torch.load(src, map_location="cpu", weights_only=False)
    assert ckpt["model_args"]["k_neighbors"] == LIGHT["k_neighbors"], "k_neighbors distinto al del checkpoint de origen"
    state = ckpt["state_dict"]
    if task == "classifier":
        state = {"feature_nets.0." + k[len("net."):]: v for k, v in state.items() if k.startswith("net.")}
    dst = Path(f"/content/init_{task}.pt")
    torch.save({"state_dict": state}, dst)
    print(f"{task}: arranque desde {src.name} (época {ckpt['extra'].get('epoch')}, model_args={ckpt['model_args']})")
    return dst

INIT = {task: warm_start_file(task) for task in TIME_BUDGET_MIN}

def make_args(task, **overrides):
    # impl="cuda": falla en lugar de caer en silencio en la implementación de referencia (lenta)
    params = dict(LIGHT, device="cuda", impl="cuda", num_modules=REGRESSOR_MODULES, init_from=INIT[task], output=None)
    params.update(overrides)
    return argparse.Namespace(task=task, **params)

print(f"Preset {PRESET}: {LIGHT} | módulos del regresor: {REGRESSOR_MODULES}")
print("Salidas:", {k: v.name for k, v in OUT_CKPT.items()})
print("Degradaciones de entrenamiento:", cfg.MAMBA_TRAIN_COMBINATIONS)

## 6. Preparar las ventanas (`prepare`)

Equivale a `python train_mamba.py prepare`, con dos ajustes para Colab:

- **Lectura por bloques del `.laz`.** `load_las_data` mantiene el objeto `laspy` completo en memoria además de las coordenadas. Con escenas de 80-100 M de puntos eso puede superar los ~12 GB de RAM de Colab gratuito. Aquí se leen solo X, Y, Z por bloques: se obtienen los mismos valores en el mismo orden, y por tanto las mismas ventanas.
- **Por escena y en caché.** Las escenas cuyas ventanas ya están en Drive se omiten (pon `FORCE_PREPARE = True` para regenerarlas, por ejemplo si cambias `MAMBA_WINDOWS_PER_SPLIT`). Cada escena conserva la semilla `SEED + i` del script original, de modo que la partición train/val/test es idéntica.

Con 40M + 80M el pico es de unos 7 GB de RAM y tarda ~10-15 min la primera vez. Si la sesión se cae por falta de RAM, ejecuta la celda de nuevo: continúa con las escenas pendientes.

In [ ]:
import gc, shutil
import numpy as np
import laspy
import psutil
from data_loader import load_las_data
from experiment import extract_patch
from models.mamba_ipf.dataset import prepare_windows

FORCE_PREPARE = False

def local_laz(terrain):
    # Copia el .laz de Drive al disco local de Colab (lectura mucho más rápida)
    dst = cfg.DATA_DIR / terrain / "points.laz"
    if not dst.exists():
        src = DRIVE_DATA / terrain / "points.laz"
        if not src.exists():
            raise FileNotFoundError(f"Falta {src}")
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    return dst

def load_xyz(path, chunk_size=5_000_000):
    # Mismo resultado que load_las_data(path)[0], sin retener el resto de atributos
    with laspy.open(path) as reader:
        xyz = np.empty((reader.header.point_count, 3), dtype=np.float64)
        n = 0
        for chunk in reader.chunk_iterator(chunk_size):
            m = len(chunk)
            xyz[n:n + m, 0], xyz[n:n + m, 1], xyz[n:n + m, 2] = chunk.x, chunk.y, chunk.z
            n += m
    return xyz[:n]

def has_windows(terrain):
    return any((DRIVE_WINDOWS / "train").glob(f"{terrain}_*.npz"))

pending = [t for t in TRAIN_TERRAINS if FORCE_PREPARE or not has_windows(t)]
if pending:
    # Caja XY del patch de evaluación de main.py (igual que en train_mamba.prepare)
    eval_points = load_las_data(local_laz(cfg.TERRAIN))[0]
    _, _, (cx, cy) = extract_patch(eval_points, cfg.PATCH_SIZE, cfg.MIN_POINTS, cfg.MAX_ATTEMPTS, cfg.PATCH_SEED)
    half = cfg.PATCH_SIZE / 2
    exclude_box = (cx - half, cy - half, cx + half, cy + half)
    del eval_points
    print(f"Patch de evaluación excluido: {exclude_box}")

    for terrain in pending:
        i = cfg.MAMBA_TRAIN_TERRAINS.index(terrain) if terrain in cfg.MAMBA_TRAIN_TERRAINS else len(cfg.MAMBA_TRAIN_TERRAINS)
        for old in DRIVE_WINDOWS.glob(f"*/{terrain}_*.npz"):
            old.unlink()
        points = load_xyz(local_laz(terrain))
        print(f"\n[{terrain}] {len(points):,} puntos | RAM usada: {psutil.virtual_memory().percent:.0f}%")
        saved, n_test = prepare_windows(
            terrain, points, DRIVE_WINDOWS, cfg.PATCH_SIZE, cfg.MIN_POINTS, cfg.MAMBA_WINDOWS_PER_SPLIT,
            cfg.MAMBA_VAL_FRACTION, cfg.MAMBA_TEST_FRACTION, exclude_box, seed=cfg.SEED + i,
        )
        print(f"[{terrain}] ventanas: train={saved['train']}, val={saved['val']} | reservadas para test: {n_test}")
        del points
        gc.collect()

for split in ("train", "val"):
    files = sorted((DRIVE_WINDOWS / split).glob("*.npz"))
    scenes = sorted({f.stem.rsplit("_", 2)[0] for f in files})
    print(f"{split}: {len(files)} ventanas de {scenes}")
extra = {f.stem.rsplit("_", 2)[0] for f in DRIVE_WINDOWS.glob("*/*.npz")} - set(TRAIN_TERRAINS)
if extra:
    print("AVISO: hay ventanas de escenas fuera de TRAIN_TERRAINS y también se usarán:", sorted(extra))

## 6b. Diagnóstico del clasificador: ¿reentrenar o solo ajustar el umbral?

Toma unos minutos y conviene ejecutarlo **antes** de gastar GPU en reentrenar el clasificador.

Durante el entrenamiento la pérdida pondera los outliers por `pos_weight` ≈ inliers/outliers (≈ 10-40 con `OUTLIER_RATIO = 0.05`). Por eso el criterio `logit > 0`, que equivale al umbral 0,5 de `MAMBA_OUTLIER_THRESHOLD`, queda sesgado hacia marcar puntos como outliers: recall alto y precisión baja. Esta celda calcula las probabilidades del clasificador sobre las ventanas de validación y muestra, para cada umbral:
- **`precision`, `recall`, `f1`**: las mismas métricas que el registro de entrenamiento.
- **`inliers_podados`**: fracción de puntos **válidos del terreno** que se eliminarían. Es la cifra que más importa para la tesis, porque cada inlier podado es terreno perdido.
- **AP** (*average precision*): calidad del ranking, independiente del umbral.

Cómo decidir:
- **Con el mejor umbral se llega a F1 ≳ 0,75 con `inliers_podados` ≲ 1 %:** el clasificador ya separa bien y solo estaba mal calibrado. Quita `"classifier"` de `TRAIN_TASKS` (paso 5) y dedica la GPU al regresor. Después pon ese umbral en `MAMBA_OUTLIER_THRESHOLD` de tu `config.py` local.
- **AP baja (≲ 0,6):** el clasificador no distingue bien y conviene reentrenarlo con el preset `a100`, que le da más contexto con parches de 1000 puntos. Al terminar, vuelve a ejecutar esta celda con `CALIBRATE = OUT_CKPT["classifier"]`.

El umbral se elige sobre validación, así que el resultado es algo optimista. La evaluación independiente es el patch de test de `main.py`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score
from models.mamba_ipf.networks import load_checkpoint

CALIBRATE = DRIVE_CKPT / "mamba_classifier.pt"   # tras reentrenar: OUT_CKPT["classifier"]
CALIB_ITEMS = 60

def classifier_scores(ckpt_path, n_items):
    model, ckpt = load_checkpoint(ckpt_path, "cuda", "cuda")
    # Mismas ventanas y semilla que la validación del entrenamiento, con el patch_size del checkpoint
    ds = tm.make_dataset("classifier", "val", n_items, False, cfg.SEED + 1, make_args("classifier", patch_size=ckpt["patch_size"]))
    probs, labels = [], []
    with torch.no_grad():
        for i in range(len(ds)):
            item = ds[i]
            probs.append(torch.sigmoid(model(item["noisy"].cuda())).flatten().cpu().numpy())
            labels.append(item["labels"].flatten().numpy() > 0.5)
    del model
    torch.cuda.empty_cache()
    return np.concatenate(probs), np.concatenate(labels)

probs, labels = classifier_scores(CALIBRATE, CALIB_ITEMS)
rows = []
for t in np.unique(np.r_[np.linspace(0.05, 0.995, 190), 0.5]):
    pred = probs > t
    tp, fp, fn = (pred & labels).sum(), (pred & ~labels).sum(), (~pred & labels).sum()
    precision, recall = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    rows.append((t, precision, recall, 2 * precision * recall / max(precision + recall, 1e-12), fp / max((~labels).sum(), 1)))
table = pd.DataFrame(rows, columns=["umbral", "precision", "recall", "f1", "inliers_podados"])

print(f"{CALIBRATE.name}: {labels.sum():,} outliers de {len(labels):,} puntos ({labels.mean():.1%}) | "
      f"AP = {average_precision_score(labels, probs):.3f}")
best = table.loc[table["f1"].idxmax()]
safe = table[table["inliers_podados"] <= 0.01]
picks = [("umbral actual (0.5)", table.loc[(table["umbral"] - 0.5).abs().idxmin()]), ("mejor F1", best)]
if len(safe):
    picks.append(("mejor F1 con ≤1% de inliers podados", safe.loc[safe["f1"].idxmax()]))
print(pd.DataFrame([r for _, r in picks], index=[n for n, _ in picks]).round(4).to_string())

## 7. Prueba corta: memoria, tiempo y épocas que caben en el presupuesto

Dos épocas cortas de cada modelo (no se guardan en Drive). La primera incluye el calentamiento de CUDA, así que se usa la segunda (columna `seconds` del registro) para extrapolar la duración de una época completa. Con eso se fija `EPOCHS[task]` para que el entrenamiento quepa en `TIME_BUDGET_MIN`.

La estimación es algo conservadora: la validación (sin gradientes) cuesta menos que el entrenamiento, pero aquí se cuenta igual. Si sale *CUDA out of memory*, baja `samples_per_item` o `patch_size` en el paso 5 y repite.

Ejecútala en cada sesión antes de entrenar: el resultado depende de la GPU que te asigne Colab.

In [ ]:
import pandas as pd
SMOKE_DIR = Path("/content/smoke")
SMOKE_ITEMS, SMOKE_VAL = 20, 5
full_epoch_items = LIGHT["items_per_epoch"] + LIGHT["val_items"]

for task in TRAIN_TASKS:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    out = SMOKE_DIR / f"{task}.pt"
    tm.train(make_args(task, epochs=2, items_per_epoch=SMOKE_ITEMS, val_items=SMOKE_VAL, output=out))
    seconds = pd.read_csv(out.with_suffix(".csv"))["seconds"].iloc[-1]
    epoch_min = seconds / (SMOKE_ITEMS + SMOKE_VAL) * full_epoch_items / 60
    EPOCHS[task] = max(1, min(LIGHT["epochs"], int(TIME_BUDGET_MIN[task] // epoch_min)))
    print(f"==> {task}: pico de memoria GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} GB | "
          f"~{epoch_min:.1f} min/época | {EPOCHS[task]} épocas ≈ {EPOCHS[task] * epoch_min:.0f} min "
          f"(presupuesto {TIME_BUDGET_MIN[task]} min)\n")
print("Épocas a usar:", EPOCHS)

## 8. Entrenar el clasificador (fase 1)

Cada vez que mejora el F1 de validación, el checkpoint se guarda en Drive en `OUT_CKPT["classifier"]` (`mamba_classifier_a100.pt` con el preset A100), y el registro de cada época queda en el `.csv` del mismo nombre. Si `"classifier"` no está en `TRAIN_TASKS`, la celda no hace nada.

⚠️ `train_mamba.py` no reanuda: si la sesión se cae, se pierde la época en curso y el mejor checkpoint guardado hasta ese momento sigue en Drive. Al volver a ejecutar la celda se empieza de cero y se sobrescriben el `.pt` y el `.csv` de salida. Con `WARM_START` el arranque se hace siempre desde el checkpoint T4, nunca desde la salida `_a100`.

Para que Colab no te desconecte por inactividad, mantén la pestaña abierta.

In [ ]:
if "classifier" in TRAIN_TASKS:
    torch.cuda.empty_cache()
    print(f"Clasificador: {EPOCHS['classifier']} épocas -> {OUT_CKPT['classifier'].name}")
    tm.train(make_args("classifier", epochs=EPOCHS["classifier"], output=OUT_CKPT["classifier"]))
else:
    print("Clasificador omitido (no está en TRAIN_TASKS)")

## 9. Entrenar el regresor (fase 2)

Es independiente del clasificador: se puede entrenar en otra sesión o en otro orden. Se guarda cada vez que baja `val_err_after_m` (error medio en metros al punto limpio más cercano después de la red). Compáralo con `val_err_before_m`, el error de la entrada ruidosa.

Ojo con lo que se puede esperar: con ruido en `xyz`, la parte del desplazamiento a lo largo del terreno no es recuperable. Incluso una proyección perfecta sobre la superficie deja `val_err_after_m` en **~0,08-0,09 m** con estas densidades (11-14 pt/m²). Ese es el piso realista, no 0. La corrida T4 llegó a 0,125 m, partiendo de 0,135 m.

In [ ]:
if "regressor" in TRAIN_TASKS:
    torch.cuda.empty_cache()
    print(f"Regresor ({REGRESSOR_MODULES} módulos): {EPOCHS['regressor']} épocas -> {OUT_CKPT['regressor'].name}")
    tm.train(make_args("regressor", epochs=EPOCHS["regressor"], output=OUT_CKPT["regressor"]))
else:
    print("Regresor omitido (no está en TRAIN_TASKS)")

## 10. Curvas de entrenamiento

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

panels = {
    "classifier": [("train_loss", "val_loss"), ("val_precision", "val_recall", "val_f1")],
    "regressor": [("train_loss", "val_loss"), ("val_err_before_m", "val_err_after_m")],
}
for path in sorted(DRIVE_CKPT.glob("mamba_*.csv")):   # corrida T4 y A100
    name = path.name
    groups = panels["classifier" if "classifier" in name else "regressor"]
    log = pd.read_csv(path)
    fig, axes = plt.subplots(1, len(groups), figsize=(12, 3.5))
    for ax, cols in zip(axes, groups):
        for col in cols:
            ax.plot(log["epoch"], log[col], marker="o", ms=3, label=col)
        ax.set_xlabel("época")
        ax.grid(alpha=0.3)
        ax.legend()
    fig.suptitle(name)
    plt.tight_layout()
    plt.show()

## 11. Llevar los checkpoints al equipo local

Los archivos ya están en `MyDrive/3d-denoising/checkpoints/`. Descárgalos desde Drive o con la celda siguiente, y cópialos en `checkpoints/` del repositorio local. Después, `python main.py` ejecutará los pipelines `("mamba", "mamba")`.

`main.py` busca los nombres `mamba_classifier.pt` y `mamba_regressor.pt` (`MAMBA_CLASSIFIER_CKPT` / `MAMBA_REGRESSOR_CKPT` en `config.py`). Para evaluar los `_a100` tienes dos opciones:
- Renombrarlos al copiarlos a `checkpoints/`.
- Apuntar esas dos variables de tu `config.py` local a los archivos nuevos.

Si ajustaste el umbral en el paso 6b, ponlo también en `MAMBA_OUTLIER_THRESHOLD`.

En local (sin CUDA) `main.py` usa la implementación de referencia de Mamba: funciona, pero la inferencia es lenta.

In [ ]:
for p in sorted(DRIVE_CKPT.glob("*.pt")):
    ckpt = torch.load(p, map_location="cpu", weights_only=False)
    print(f"{p.name}: kind={ckpt['kind']} | model_args={ckpt['model_args']} | patch_size={ckpt['patch_size']} | "
          f"época={ckpt['extra'].get('epoch')} | {ckpt['extra'].get('monitor')}={ckpt['extra'].get('val', {}).get(ckpt['extra'].get('monitor'))}")

# Descarga directa al navegador (opcional):
# from google.colab import files
# for p in DRIVE_CKPT.glob("mamba_*.*"): files.download(str(p))

# Evaluación: SOR + MLS frente a 3DMambaIPF

Esta sección reutiliza las funciones de `main.py` (`run_outlier_removal`, `run_displacement`, `evaluate`, `outlier_detection_metrics`) para comparar los pipelines con validez estadística:

- **Repeticiones.** El mismo patch de evaluación se degrada con **N semillas de ruido distintas**. Cada semilla produce la misma nube corrompida para todos los pipelines, así que la comparación es **pareada**. La primera semilla es `NOISE_SEED` de `config.py`, de modo que coincide con lo que produce `main.py`.
- **Patches de evaluación.**
  - **Test:** el patch de `main.py` en `Sunset_Crater_..._1M`, excluido del entrenamiento en `prepare`.
  - **Fuera de distribución (opcional):** un patch de `San_Isabel_NF_CO_Debris_1M`, una escena que no se usó para entrenar. Para incluirlo, sube `data/San_Isabel_NF_CO_Debris_1M/points.laz` (~3 MB) a Drive.
  - No se usan otros patches de Sunset Crater o Northern California: podrían caer en ventanas que el modelo vio al entrenar.
- **Pipelines.** `SOR + MLS` (referencia), `Mamba + Mamba` (propuesta) y las dos ablaciones (`SOR + Mamba-reg`, `Mamba-cls + MLS`), que muestran qué fase aporta la mejora.
- **Prueba estadística.** Wilcoxon de rangos con signo pareado (no paramétrico), con corrección de Holm por las 6 métricas de cada comparación. Además se reportan el tamaño del efecto (diferencia mediana, también relativa) y el % de semillas en que el método mejora a la referencia.

Pasos a ejecutar antes, en una sesión nueva: 1-5. `TRAIN_TASKS` no influye aquí.

Los resultados se guardan en Drive, en `evaluation/<checkpoints>_thr<umbral>/`, una fila por semilla. Si la sesión se cae, la celda E2 continúa donde se quedó. Si cambias los checkpoints o el umbral, los resultados van a otra carpeta y no se mezclan.

## E1. Configuración de la evaluación

In [ ]:
import contextlib, io, shutil, subprocess, sys, time, types
import numpy as np
import pandas as pd

try:
    import open3d  # noqa: F401  (visualization.py lo importa; aquí no se abre el visor 3D)
except ImportError:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open3d"])
        import open3d  # noqa: F401
    except Exception:
        sys.modules["open3d"] = types.ModuleType("open3d")
        print("open3d no disponible: se sustituye por un módulo vacío (solo lo usa el visor 3D, desactivado aquí)")

import main as pipeline_main
from data_loader import load_las_data
from experiment import extract_patch, degrade_patch, experiment_name, describe_degradation
from metrics.evaluation import evaluate
from metrics.outlier_detection import outlier_detection_metrics

def pick_ckpt(task):
    # Prioriza los checkpoints A100; si no existen, los de la corrida T4
    for name in (f"mamba_{task}_a100.pt", f"mamba_{task}.pt"):
        if (DRIVE_CKPT / name).exists():
            return DRIVE_CKPT / name
    raise FileNotFoundError(f"No hay checkpoint de {task} en {DRIVE_CKPT}")

cfg.MAMBA_CLASSIFIER_CKPT = pick_ckpt("classifier")
cfg.MAMBA_REGRESSOR_CKPT = pick_ckpt("regressor")
cfg.MAMBA_OUTLIER_THRESHOLD = 0.91      # umbral elegido en el paso 6b
cfg.MAMBA_DEVICE, cfg.MAMBA_IMPL = "cuda", "cuda"
cfg.MAMBA_BATCH_SIZE = 32               # parches por lote en inferencia (8 en config.py; la GPU admite más)
cfg.SHOW_FIGURES, cfg.OPEN3D_VIEWER = False, False

cfg.PIPELINES = [("sor", "mls"), ("mamba", "mamba"), ("sor", "mamba"), ("mamba", "mls")]
BASELINE, PROPOSED = ("sor", "mls"), ("mamba", "mamba")
N_SEEDS = 20
SEEDS = [cfg.NOISE_SEED + i for i in range(N_SEEDS)]
EVAL_COMBINATIONS = cfg.NOISE_COMBINATIONS      # p. ej. [["gaussian", "global_outliers"], ["gaussian", "laplacian", "clustered_outliers"]]

def eval_laz(scene):
    dst = cfg.DATA_DIR / scene / "points.laz"
    if not dst.exists() and (DRIVE_DATA / scene / "points.laz").exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(DRIVE_DATA / scene / "points.laz", dst)
    return dst

EVAL_SCENES = {
    cfg.TERRAIN: "test",                                  # patch de main.py, excluido del entrenamiento
    "San_Isabel_NF_CO_Debris_1M": "fuera de distribución",  # escena no usada para entrenar (opcional)
}
EVAL_SCENES = {s: role for s, role in EVAL_SCENES.items() if eval_laz(s).exists()}

PIPELINES = pipeline_main.available_pipelines()
LABELS = {p: pipeline_main.pipeline_label(*p) for p in PIPELINES}
BASE_LABEL, PROP_LABEL = LABELS[BASELINE], LABELS[PROPOSED]
REFERENCE = "Corrompida"
# Colores categóricos en orden fijo (validados para daltonismo); la referencia sin filtrar, en gris
COLORS = dict(zip(LABELS.values(), ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]), **{REFERENCE: "#9a9a94"})
PHASE1_COLORS = {"SOR": "#2a78d6", "Mamba-cls": "#eb6834"}

EVAL_NAME = f"{cfg.MAMBA_CLASSIFIER_CKPT.stem}+{cfg.MAMBA_REGRESSOR_CKPT.stem}_thr{cfg.MAMBA_OUTLIER_THRESHOLD}"
EVAL_DIR = DRIVE_ROOT / "evaluation" / EVAL_NAME
EVAL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV, DETECTION_CSV = EVAL_DIR / "metricas_por_semilla.csv", EVAL_DIR / "deteccion_por_semilla.csv"

print("Clasificador:", cfg.MAMBA_CLASSIFIER_CKPT.name, "| umbral", cfg.MAMBA_OUTLIER_THRESHOLD)
print("Regresor:", cfg.MAMBA_REGRESSOR_CKPT.name)
print("Pipelines:", list(LABELS.values()))
print("Escenas:", EVAL_SCENES)
print(f"{N_SEEDS} semillas × {len(EVAL_COMBINATIONS)} combinaciones de ruido | resultados en {EVAL_DIR}")

## E2. Ejecutar los pipelines con N semillas

Por cada semilla: degrada el patch, ejecuta cada pipeline y calcula las métricas de `metrics/evaluation.py` frente al patch limpio. Las métricas se calculan sobre:
- **La nube completa** de cada pipeline, que incluye los outliers que sobreviven a la fase 1.
- **Solo los inliers** (filas "(inliers)" de `main.py`), que aíslan el efecto del suavizado.

También guarda la detección de la fase 1 (SOR frente a Mamba-cls) frente a los outliers inyectados. La fase 1 se calcula una sola vez por método y se reutiliza en los pipelines que la comparten.

El tiempo por semilla se imprime en la primera iteración. Con ~180 k puntos por patch, cuenta con unos minutos por semilla, sobre todo por MLS y las métricas, que corren en CPU.

In [ ]:
def append_rows(path, rows):
    pd.DataFrame(rows).to_csv(path, mode="a", header=not path.exists(), index=False)

def done_keys(path):
    if not path.exists():
        return set()
    df = pd.read_csv(path)
    return set(zip(df["escena"], df["combinacion"], df["semilla"]))

def run_seed(patch_clean, components, seed):
    # Devuelve {nube: puntos}, {método de fase 1: métricas de detección} y lo necesario para las figuras
    corrupted, is_outlier, _ = degrade_patch(patch_clean, components, cfg.NOISE_SIGMA, cfg.NOISE_LAPLACE_SCALE,
                                             cfg.NOISE_AXES, cfg.OUTLIER_RATIO, seed, verbose=False)
    clouds = {REFERENCE: corrupted}
    if is_outlier.any():
        clouds[f"{REFERENCE} (inliers)"] = corrupted[~is_outlier]
    phase1, outputs = {}, {}
    with contextlib.redirect_stdout(io.StringIO()):          # silencia los mensajes de main.py
        for p1, p2 in PIPELINES:
            if p1 not in phase1:
                phase1[p1] = pipeline_main.run_outlier_removal(p1, corrupted)
            kept, kept_mask = phase1[p1]
            denoised = pipeline_main.run_displacement(p2, kept)
            remaining_outlier = is_outlier[kept_mask]
            clouds[LABELS[(p1, p2)]] = denoised
            if is_outlier.any():
                clouds[f"{LABELS[(p1, p2)]} (inliers)"] = denoised[~remaining_outlier]
            outputs[(p1, p2)] = (kept_mask, denoised, remaining_outlier)
    detection = {
        pipeline_main.PHASE1_LABELS[p1]: outlier_detection_metrics(~kept_mask, is_outlier)
        for p1, (_, kept_mask) in phase1.items() if p1 != "none" and is_outlier.any()
    }
    return clouds, detection, corrupted, is_outlier, outputs

for scene, role in EVAL_SCENES.items():
    points = load_las_data(eval_laz(scene))[0]
    patch_clean, _, center = extract_patch(points, cfg.PATCH_SIZE, cfg.MIN_POINTS, cfg.MAX_ATTEMPTS, cfg.PATCH_SEED)
    del points
    print(f"\n[{scene}] ({role}) patch de {cfg.PATCH_SIZE:.0f} m en {np.round(center, 1)}: {len(patch_clean):,} puntos")
    for components in EVAL_COMBINATIONS:
        combo = experiment_name(components)
        done = done_keys(RESULTS_CSV)
        for seed in SEEDS:
            if (scene, combo, seed) in done:
                continue
            start = time.time()
            clouds, detection, *_ = run_seed(patch_clean, components, seed)
            keys = {"escena": scene, "rol": role, "combinacion": combo, "semilla": seed}
            append_rows(RESULTS_CSV, [
                {**keys, "nube": name, **evaluate(cloud, patch_clean, cfg.DCD_ALPHA, cfg.NORMAL_K_NEIGHBORS)}
                for name, cloud in clouds.items()
            ])
            if detection:
                append_rows(DETECTION_CSV, [{**keys, "metodo": m, **values} for m, values in detection.items()])
            print(f"  {combo} | semilla {seed}: {time.time() - start:.0f} s")
print("\nEvaluación completa:", RESULTS_CSV)

## E3. Tablas: estadística descriptiva y Wilcoxon pareado

- **`descriptiva.csv`:** media ± desviación estándar y mediana de cada métrica, por escena, combinación de ruido, subconjunto (nube completa o solo inliers) y pipeline.
- **`wilcoxon_metricas.csv`:** cada pipeline frente a `SOR + MLS`, con estas columnas:
  - `mediana_ref` y `mediana_metodo`.
  - `dif_mediana`: mediana de las diferencias pareadas; negativa significa mejor, porque todas las métricas de calidad geométrica son "menor es mejor".
  - `dif_rel_%`: la misma diferencia, relativa a la mediana de la referencia.
  - `mejora_%`: porcentaje de semillas en que el método mejora a la referencia.
  - `p` y `p_holm`: p-valor de Wilcoxon, sin corregir y corregido por Holm sobre las 6 métricas de cada comparación.
- **`wilcoxon_deteccion.csv`:** Mamba-cls frente a SOR en precisión, recall y F1 (mayor es mejor) y en inliers eliminados (menor es mejor).

Con N = 20 semillas, Wilcoxon puede detectar diferencias consistentes. Un `p_holm` < 0,05 junto con `mejora_%` cercano a 100 indica una mejora robusta. Reporta siempre el tamaño del efecto: un p-valor pequeño con `dif_rel_%` de −0,5 % no es una mejora práctica.

In [ ]:
from scipy.stats import wilcoxon

METRICS = ["RMSE_z (m)", "CD (m)", "DCD", "HD (m)", "HD95 (m)", "UNAE (°)"]
DET_METRICS = {"Precisión": "max", "Recall": "max", "F1": "max", "Inliers elim. (%)": "min"}
GROUP = ["escena", "rol", "combinacion"]

res = pd.read_csv(RESULTS_CSV)
inliers = res["nube"].str.endswith(" (inliers)")
res["subconjunto"] = np.where(inliers, "solo inliers", "nube completa")
res["metodo"] = res["nube"].str.replace(" (inliers)", "", regex=False)
det = pd.read_csv(DETECTION_CSV) if DETECTION_CSV.exists() else pd.DataFrame()

def holm(pvalues):
    p = np.asarray(pvalues, dtype=float)
    adjusted, running = np.empty_like(p), 0.0
    for rank, i in enumerate(np.argsort(p)):
        running = max(running, min(1.0, (len(p) - rank) * p[i]))
        adjusted[i] = running
    return adjusted

def paired_tests(df, group_cols, metrics, reference, better):
    # better: {métrica: "min" | "max"}; compara cada método con la referencia, semilla a semilla
    rows = []
    for keys, sub in df.groupby(group_cols):
        keys = dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))
        methods = [m for m in sub["metodo"].unique() if m not in (reference, REFERENCE)]
        for method in methods:
            block = []
            for metric in metrics:
                wide = sub.pivot_table(index="semilla", columns="metodo", values=metric).dropna(subset=[reference, method])
                ref, val = wide[reference].to_numpy(), wide[method].to_numpy()
                diff = val - ref
                improves = diff < 0 if better[metric] == "min" else diff > 0
                p = wilcoxon(val, ref).pvalue if np.any(diff != 0) and len(diff) >= 6 else np.nan
                block.append({**keys, "comparacion": f"{method} vs {reference}", "metrica": metric, "n": len(diff),
                              "mediana_ref": np.median(ref), "mediana_metodo": np.median(val),
                              "dif_mediana": np.median(diff), "dif_rel_%": 100 * np.median(diff) / abs(np.median(ref)),
                              "mejora_%": 100 * improves.mean(), "p": p})
            p_holm = holm([np.nan_to_num(r["p"], nan=1.0) for r in block])
            for r, ph in zip(block, p_holm):
                r["p_holm"] = ph if not np.isnan(r["p"]) else np.nan
            rows += block
    return pd.DataFrame(rows)

# Descriptiva
desc = res.groupby(GROUP + ["subconjunto", "metodo"])[METRICS].agg(["mean", "std", "median"])
desc.to_csv(EVAL_DIR / "descriptiva.csv")
pretty = res.groupby(GROUP + ["subconjunto", "metodo"])[METRICS].agg(
    lambda s: f"{s.mean():.4f} ± {s.std():.4f}").reset_index()

# Wilcoxon de las métricas geométricas (todas: menor es mejor)
tests = paired_tests(res, GROUP + ["subconjunto"], METRICS, BASE_LABEL, {m: "min" for m in METRICS})
tests.to_csv(EVAL_DIR / "wilcoxon_metricas.csv", index=False)

# Wilcoxon de la detección de la fase 1
det_tests = pd.DataFrame()
if len(det):
    det_tests = paired_tests(det, GROUP, list(DET_METRICS), "SOR", DET_METRICS)
    det_tests.to_csv(EVAL_DIR / "wilcoxon_deteccion.csv", index=False)

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
print("== Media ± desviación estándar ==")
display(pretty)
print(f"\n== {PROP_LABEL} vs {BASE_LABEL} (Wilcoxon pareado, Holm) ==")
cols = GROUP[:1] + ["subconjunto", "metrica", "mediana_ref", "mediana_metodo", "dif_rel_%", "mejora_%", "p_holm"]
display(tests[tests["comparacion"] == f"{PROP_LABEL} vs {BASE_LABEL}"][cols].round(4))
if len(det_tests):
    print("\n== Fase 1: Mamba-cls vs SOR ==")
    display(det_tests[GROUP[:1] + ["metrica", "mediana_ref", "mediana_metodo", "mejora_%", "p_holm"]].round(4))
print(f"\nTablas completas (todas las comparaciones y ablaciones) en {EVAL_DIR}")

## E4. Figuras comparativas

Las figuras se guardan en PNG a 200 dpi en la carpeta de resultados, listas para la tesis:

1. **`boxplots_<escena>_<ruido>_<subconjunto>.png`:** un panel por métrica y una caja por pipeline, con la nube corrompida en gris como referencia. Sobre cada caja aparece el `p_holm` frente a `SOR + MLS`. Es la figura principal: muestra a la vez la diferencia y la variabilidad entre semillas.
2. **`diferencias_pareadas_<escena>_<ruido>.png`:** diferencia relativa de `Mamba + Mamba` frente a `SOR + MLS`, semilla a semilla. Bajo cero significa que Mamba es mejor; sobre cada métrica se indica el % de semillas en que mejora.
3. **`deteccion_<escena>_<ruido>.png`:** fase 1, SOR frente a Mamba-cls (media ± desviación estándar entre semillas).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.transforms import blended_transform_factory

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#8a8a85",
                     "axes.labelcolor": "#3b3b38", "xtick.color": "#5c5c58", "ytick.color": "#5c5c58",
                     "axes.titlesize": 10, "font.size": 9})
TEXT, MUTED = "#2b2b29", "#6b6b66"

def short(label):
    return label.replace(" + ", "\n+ ")

def p_text(p):
    return "" if np.isnan(p) else ("p<0.001" if p < 0.001 else f"p={p:.3f}")

def fname(*parts):
    return "_".join(str(p).replace(" ", "-").replace("+", "_").replace("(", "").replace(")", "") for p in parts) + ".png"

for (scene, role, combo), sub in res.groupby(GROUP):
    for subset, data in sub.groupby("subconjunto"):
        order = [REFERENCE] + [m for m in LABELS.values() if m in set(data["metodo"])]
        fig, axes = plt.subplots(2, 3, figsize=(13, 7.5), constrained_layout=True)
        for ax, metric in zip(axes.flat, METRICS):
            values = [data.loc[data["metodo"] == m, metric].to_numpy() for m in order]
            box = ax.boxplot(values, patch_artist=True, widths=0.55, showfliers=True,
                             medianprops=dict(color=TEXT, linewidth=1.5), whiskerprops=dict(color=MUTED),
                             capprops=dict(color=MUTED), flierprops=dict(marker="o", markersize=3, markerfacecolor=MUTED,
                                                                         markeredgecolor="none"))
            for patch, m in zip(box["boxes"], order):
                patch.set(facecolor=COLORS[m], edgecolor="white", linewidth=2, alpha=0.9)
            ax.set_xticks(range(1, len(order) + 1), [short(m) for m in order], fontsize=7.5)
            ax.set_title(metric, color=TEXT, pad=16)    # deja sitio a los p-valores sobre las cajas
            ax.grid(axis="y", alpha=0.25)
            trans = blended_transform_factory(ax.transData, ax.transAxes)
            sel = tests[(tests["escena"] == scene) & (tests["combinacion"] == combo) &
                        (tests["subconjunto"] == subset) & (tests["metrica"] == metric)]
            for x, m in enumerate(order, start=1):
                row = sel[sel["comparacion"] == f"{m} vs {BASE_LABEL}"]
                label = "ref." if m == BASE_LABEL else (p_text(row["p_holm"].iloc[0]) if len(row) else "")
                ax.text(x, 1.0, label, transform=trans, ha="center", va="bottom", fontsize=7, color=MUTED)
        fig.suptitle(f"{scene} ({role}) | {combo} | {subset} | {data['semilla'].nunique()} semillas\n"
                     f"p de Wilcoxon pareado (Holm) frente a {BASE_LABEL}", color=TEXT)
        fig.savefig(EVAL_DIR / fname("boxplots", scene, combo, subset), dpi=200)
        plt.show()

    # Diferencias pareadas de la propuesta frente a la referencia (nube completa)
    full = sub[sub["subconjunto"] == "nube completa"]
    wide = {metric: full.pivot_table(index="semilla", columns="metodo", values=metric) for metric in METRICS}
    if PROP_LABEL in wide[METRICS[0]] and BASE_LABEL in wide[METRICS[0]]:
        rng = np.random.default_rng(0)
        fig, ax = plt.subplots(figsize=(10, 4.2), constrained_layout=True)
        ax.axhline(0, color=MUTED, linewidth=1)
        for i, metric in enumerate(METRICS):
            w = wide[metric]
            rel = 100 * (w[PROP_LABEL] - w[BASE_LABEL]) / w[BASE_LABEL].abs()
            ax.scatter(i + rng.uniform(-0.15, 0.15, len(rel)), rel, s=36, color=COLORS[PROP_LABEL],
                       edgecolor="white", linewidth=1, zorder=3)
            ax.plot([i - 0.25, i + 0.25], [rel.median()] * 2, color=TEXT, linewidth=2, zorder=4)
            ax.text(i, 1.0, f"mejora en {100 * (rel < 0).mean():.0f}%", transform=blended_transform_factory(ax.transData, ax.transAxes),
                    ha="center", va="bottom", fontsize=8, color=MUTED)
        ax.set_xticks(range(len(METRICS)), METRICS)
        ax.set_ylabel(f"diferencia relativa vs {BASE_LABEL} (%)\nnegativo = mejor")
        ax.grid(axis="y", alpha=0.25)
        ax.set_title(f"{PROP_LABEL} frente a {BASE_LABEL}, semilla a semilla (línea: mediana)\n{scene} | {combo}",
                     color=TEXT, pad=18)
        fig.savefig(EVAL_DIR / fname("diferencias_pareadas", scene, combo), dpi=200)
        plt.show()

# Detección de la fase 1
if len(det):
    for (scene, role, combo), sub in det.groupby(GROUP):
        methods = [m for m in PHASE1_COLORS if m in set(sub["metodo"])]
        stats = sub.groupby("metodo")[list(DET_METRICS)].agg(["mean", "std"])
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8), width_ratios=[3, 1], constrained_layout=True)
        width = 0.36
        for j, m in enumerate(methods):
            offset = (j - (len(methods) - 1) / 2) * (width + 0.04)
            for ax, cols in ((ax1, ["Precisión", "Recall", "F1"]), (ax2, ["Inliers elim. (%)"])):
                x = np.arange(len(cols)) + offset
                means = [stats.loc[m, (c, "mean")] for c in cols]
                stds = [stats.loc[m, (c, "std")] for c in cols]
                ax.bar(x, means, width, yerr=stds, color=PHASE1_COLORS[m], edgecolor="white", linewidth=2,
                       error_kw=dict(ecolor=MUTED, capsize=3, elinewidth=1), label=m)
                for xi, v, s in zip(x, means, stds):
                    ax.text(xi, v + np.nan_to_num(s), f"{v:.2f}", ha="center", va="bottom", fontsize=7.5, color=TEXT)
        ax1.set_xticks(range(3), ["Precisión", "Recall", "F1"])
        ax1.set_ylim(0, 1.08)
        ax2.set_xticks([0], ["Inliers eliminados (%)"])
        for ax in (ax1, ax2):
            ax.grid(axis="y", alpha=0.25)
        ax1.legend(frameon=False, loc="upper left")
        fig.suptitle(f"Fase 1: detección de outliers (media ± d.e. entre semillas)\n{scene} | {combo}", color=TEXT)
        fig.savefig(EVAL_DIR / fname("deteccion", scene, combo), dpi=200)
        plt.show()
print("Figuras guardadas en", EVAL_DIR)

## E5. Figuras cualitativas y curva precisión-recall

Para una semilla representativa (la primera, que coincide con `main.py`):
- **Comparación visual de cada pipeline:** `plot_denoising_comparison` de `visualization.py`, la misma figura que genera `main.py`, sobre el patch de test.
- **Curva precisión-recall del clasificador Mamba en el patch de test:**
  - el punto de operación con `MAMBA_OUTLIER_THRESHOLD`, en naranja;
  - el de SOR, en azul.

  SOR no produce probabilidades, así que es un único punto. Si ese punto queda por debajo de la curva de Mamba, existe un umbral de Mamba que mejora a SOR en precisión y recall a la vez.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
from visualization import plot_denoising_comparison
from models.mamba_ipf.inference import mamba_outlier_removal

QUAL_SCENE, QUAL_SEED, QUAL_COMPONENTS = cfg.TERRAIN, SEEDS[0], EVAL_COMBINATIONS[0]
points = load_las_data(eval_laz(QUAL_SCENE))[0]
patch_clean, _, _ = extract_patch(points, cfg.PATCH_SIZE, cfg.MIN_POINTS, cfg.MAX_ATTEMPTS, cfg.PATCH_SEED)
del points
_, detection, corrupted, is_outlier, outputs = run_seed(patch_clean, QUAL_COMPONENTS, QUAL_SEED)
combo = experiment_name(QUAL_COMPONENTS)
noise_label = describe_degradation(QUAL_COMPONENTS, cfg.NOISE_SIGMA, cfg.NOISE_LAPLACE_SCALE, cfg.OUTLIER_RATIO)

# 1. Comparación visual de cada pipeline (mismo submuestreo que main.py)
for (p1, p2), (kept_mask, denoised, remaining_outlier) in outputs.items():
    rng_vis = np.random.default_rng(123)
    i_clean = rng_vis.choice(len(patch_clean), min(cfg.MAX_DISPLAY, len(patch_clean)), replace=False)
    i_corr = rng_vis.choice(len(corrupted), min(cfg.MAX_DISPLAY, len(corrupted)), replace=False)
    i_den = rng_vis.choice(len(denoised), min(cfg.MAX_DISPLAY, len(denoised)), replace=False)
    plot_denoising_comparison(patch_clean[i_clean], corrupted[i_corr], is_outlier[i_corr], denoised[i_den],
                              remaining_outlier[i_den], noise_label, EVAL_DIR / fname("cualitativa", p1, p2, combo),
                              method_label=LABELS[(p1, p2)], show=True)

# 2. Curva precisión-recall del clasificador en el patch de test
if is_outlier.any() and "Mamba-cls" in detection:
    _, _, probabilities, _ = mamba_outlier_removal(
        corrupted, cfg.MAMBA_CLASSIFIER_CKPT, threshold=cfg.MAMBA_OUTLIER_THRESHOLD, seed_k=cfg.MAMBA_SEED_K,
        batch_size=cfg.MAMBA_BATCH_SIZE, device=cfg.MAMBA_DEVICE, mamba_impl=cfg.MAMBA_IMPL)
    precision, recall, _ = precision_recall_curve(is_outlier, probabilities)
    ap = average_precision_score(is_outlier, probabilities)
    fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
    ax.plot(recall, precision, color=PHASE1_COLORS["Mamba-cls"], linewidth=2, label=f"Mamba-cls (AP = {ap:.3f})")
    for method, marker in (("Mamba-cls", "o"), ("SOR", "s")):
        if method in detection:
            d = detection[method]
            label = f"{method} (umbral {cfg.MAMBA_OUTLIER_THRESHOLD})" if method == "Mamba-cls" else "SOR"
            ax.scatter(d["Recall"], d["Precisión"], s=90, marker=marker, color=PHASE1_COLORS[method],
                       edgecolor="white", linewidth=2, zorder=3, label=label)
    ax.set(xlim=(0, 1.02), ylim=(0, 1.02), xlabel="Recall", ylabel="Precisión")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, loc="lower left")
    ax.set_title(f"Fase 1 en el patch de test | {combo} | semilla {QUAL_SEED}", color=TEXT)
    fig.savefig(EVAL_DIR / fname("precision_recall", QUAL_SCENE, combo), dpi=200)
    plt.show()